In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

# IMPORTING NECESSARY LIBRARIES

# Data Visualisation
import matplotlib.pyplot as plt
import seaborn as sns

# Data Pre-Processing
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Train-Test Splitting and Hyperparameter Tuning
from sklearn.model_selection import train_test_split, RandomizedSearchCV

# Models
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier

from sklearn.metrics import f1_score

In [ ]:
train_df = pd.read_csv("/kaggle/input/competitions/workstation-rental-price-tier-prediction/kaggle_ready_dataset/train.csv")
test_df = pd.read_csv("/kaggle/input/competitions/workstation-rental-price-tier-prediction/kaggle_ready_dataset/test.csv")

# 1. INITIAL OVERVIEW

In [ ]:
train_df.head()

In [ ]:
train_df.info()

# 2. EXPLORATORY DATA ANALYSIS

## a. Target Variable Distribution (in Percentages)

In [ ]:
train_df["price_tier"].value_counts(normalize = True).sort_index().mul(100).round(2)

Relatively balanced, so no need to under/over-sample (class rebalancing).

## b. Missing Values

In [ ]:
drop_cols = []

In [ ]:
train_df.isna().any().sum()

In [ ]:
train_df.isna().any(axis = 1).sum()

In [ ]:
train_df.isna().sum().sum()

In [ ]:
# Missing count and percentage per column
missing = train_df.isna().sum()

missing_summary = (
    missing[missing > 0]
    .to_frame("missing_count")
    .assign(
        missing_pct = lambda x:
        x["missing_count"] / len(train_df) * 100
    )
    .sort_values("missing_pct", ascending=False)
)

missing_summary

As the column with largest missingness, borough is investigated for whether or not it is safe to drop

In [ ]:
train_df.assign(borough_missing = train_df["borough"].isna()).groupby("price_tier")["borough_missing"].mean().mul(100)

Missingness is relatively uniform across class. Hence, we may drop borough in preprocessing.

In [ ]:
drop_cols.append('borough')

## c. Duplicates

In [ ]:
train_df.duplicated().sum()

No duplicates are dropped. The space_id feature uniquely identifies a real-world entity (the workspace), and hence need not be dropped while checking for duplicates.

## d. Incorrect/Impossible Values

In [ ]:
train_df.describe().T

Many latitudes are above +90, when the range of latitude is [-90, +90].

In [ ]:
len(train_df[train_df["geo_lat"].between(-90, 90)])

In [ ]:
invalid_lat = ~train_df["geo_lat"].between(-90, 90)

train_df.assign(invalid_lat = invalid_lat).groupby("metro")["invalid_lat"].agg(["count", "mean"]).sort_values("mean", ascending = False)

Invalid latitude entries are associated with 9 metros FOR SURE, while to a very high degree with another.
This indicates a systematic issue pertaining to THOSE metros.
For now, we will exclude geo_lat from modelling due to its inaccuracy.

In [ ]:
drop_cols.append("geo_lat")

In [ ]:
cat_cols = train_df.select_dtypes(include=["object"]).columns.drop(["borough"])

for col in cat_cols:
    print(f"\n--- {col} ---")
    print("Unique values:", train_df[col].nunique())
    print(train_df[col].value_counts(dropna = False).head(10))

## e. Numerical Features

### i. Summary Stats

In [ ]:
train_df.describe().T

The differences in ranges indicate the need for the use of a scaler.

### ii. Histograms

In [ ]:
num_cols = train_df.select_dtypes(include = ["number"]).columns.drop(["price_tier", "geo_lat"])

train_df[num_cols].hist(figsize = (16, 12), bins = 30)

Most operator-related features look approximately uniform (operator_portfolio_size and the operator_metro features are right-skewed).

Workspace specification features, such as max_seats and zones, are concentrated at small values with right-skewed distributions.

Term-day, floor, and ceiling features are concentrated at smaller values or exhibit little variation.

Open-slot features exhibit concentration at both ends of their ranges, with distributions becoming more left-skewed going from a to e.

Review-count features have relatively concentrated distributions.

Score-related features are concentrated at smaller values, with right-skewed distributions. 


Note to Self: Right-Skewed => Many values on the left side, fall down going right. And vice versa.

### iii. Outlier Handling

In [ ]:
outlier_count_by_feature = {}

for feature in num_cols:
    q1 = train_df[feature].quantile(0.25)
    q3 = train_df[feature].quantile(0.75)
    iqr = q3 - q1

    lower_criteria = (train_df[feature] < (q1 - 1.5 * iqr))
    upper_criteria = (train_df[feature] > (q3 + 1.5 * iqr))
    outliers = (train_df.loc[lower_criteria | upper_criteria])

    outlier_count_by_feature[feature] = len(outliers)

outlier_summary = pd.DataFrame({
    "feature": num_cols,
    "outlier_count": [outlier_count_by_feature[f] for f in num_cols]
})

outlier_summary["outlier_pct"] = (outlier_summary["outlier_count"] / len(train_df) * 100)

outlier_summary = outlier_summary.sort_values("outlier_pct", ascending = False).reset_index(drop = True)

outlier_summary.round({"outlier_pct": 2})

In [ ]:
drop_cols

## f. Categorical Features

In [ ]:
cat_summary = pd.DataFrame({
    "unique_values": train_df[cat_cols].nunique(),
    "missing_count": train_df[cat_cols].isna().sum()
})

cat_summary.sort_values("unique_values", ascending = False)

facilities is a list of "FCL-xxx" -> Multi-Hot Encode
Low-cardinality (<= 16) -> OneHot
Higher-cardinality -> Require further investigation on best method.

## g. Numerical Feature-to-Feature and Feature-to-Target Relationships

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

corr_cols = train_df.select_dtypes(include="number").columns.drop(
    ["space_id", "geo_lat"], errors = "ignore"
)

plt.figure(figsize = (14, 10))
sns.heatmap(train_df[corr_cols].corr(), cmap = "coolwarm", center=0)
plt.show()

max_seats, zones, utility_rooms, workstations -> Strong feature to target.

max_seats, utility_rooms, zones, and workstations are positively correlated.
min_term and max_term features are positively correlated within their groups.
open_slots features are strongly correlated.

Review counts are strongly correlated. 
Score features are strongly correlated.
operator_metro features are strongly correlated, though to varying degrees.

All of these are strong, positive correlations.

## h. Categorical Feature-to-Target Relationships

In [ ]:
# 15 most freq space_classes

top_classes = train_df["space_class"].value_counts().head(15).index

subset = train_df[train_df["space_class"].isin(top_classes)]

pd.crosstab(
    subset["space_class"],
    subset["price_tier"],
    normalize = "index"
).plot(kind = "bar", stacked = True, figsize = (12, 5))

In [ ]:
cat_cols = ["metro", "access_type", "compliance_status", "operator_class", 
        "operator_id_verified", "operator_has_avatar", "operator_certified", "bookable"
]

fig, axes = plt.subplots(4, 2, figsize = (14, 16))
axes = axes.flatten()

for i, col in enumerate(cat_cols):
    pd.crosstab(train_df[col].fillna("Missing"), train_df["price_tier"], normalize = "index"
    ).plot(kind = "bar", stacked = True, ax = axes[i], legend = (i == 0))

    axes[i].set_title(f"{col} vs price_tier")
    axes[i].set_ylabel("Proportion")
    axes[i].set_xlabel("")

for j in range(len(cat_cols), len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.show()

 metro, access_type, operator_class, compliance_status are useful features

In [ ]:
# space_id is an irrelevant column here
# operator_ref is also irrelevant for the most part.
drop_cols.extend(['space_id', 'operator_ref'])

X = train_df.drop(columns = ["price_tier"] + drop_cols, errors = "ignore")
y = train_df["price_tier"]

X_test = test_df.drop(columns=drop_cols, errors = "ignore")

# 3. PRE-PROCESSING

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size = 0.2, random_state = 42, stratify = y)

## FEATURE ENGINEERING AND SELECTION

## a. Handling facilities

In [ ]:
import ast
from sklearn.preprocessing import MultiLabelBinarizer # multi-hot encoding

# Safely parse and deduplicate facility lists
def parse_facilities(x):
    if isinstance(x, str):
        try:
            x = ast.literal_eval(x)
        except (ValueError, SyntaxError):
            return []

    if isinstance(x, (list, tuple, set)):
        return list(dict.fromkeys(x))

    return []

for df in [X_train, X_val, X_test]:
    df["facilities"] = df["facilities"].apply(parse_facilities)

# Facility Count
for df in [X_train, X_val, X_test]:
    df["facility_count"] = df["facilities"].apply(len)

# Fit ONLY on training data
mlb = MultiLabelBinarizer()
mlb.fit(X_train["facilities"])

# Transform all three datasets
for df in [X_train, X_val, X_test]:
    facility_matrix = mlb.transform(df["facilities"])

    facility_df = pd.DataFrame(
        facility_matrix,
        columns=[f"facility_{c}" for c in mlb.classes_],
        index=df.index,
        dtype="uint8"
    )

    df.drop(columns=["facilities"], inplace=True)
    df[facility_df.columns] = facility_df

In [ ]:
X_train['facility_count']

## b. Handling Date Columns

In [ ]:
date_cols = ["snapshot_start_date", "snapshot_end_date", "first_review_date", "last_review_date"]

for df in [X_train, X_val, X_test]:
    for col in date_cols:
        df[col] = pd.to_datetime(df[col], errors = "coerce")

    df["snapshot_interval_days"] = (df["snapshot_end_date"] - df["snapshot_start_date"]).dt.days
    df["review_interval_days"] = (df["last_review_date"] - df["first_review_date"]).dt.days

    df.drop(columns=date_cols, inplace=True)

## c. Group-based Feature Engineering

In [ ]:
# Count categories using training data only
class_counts = X_train["space_class"].fillna("Missing").value_counts()

zone_counts = X_train["zone"].astype("string").fillna("Missing").value_counts()

metro_counts = X_train["metro"].astype("string").fillna("Missing").value_counts()

# Map counts onto all datasets
for df in [X_train, X_val, X_test]:
    df["class_count"] = df["space_class"].fillna("Missing").map(class_counts).fillna(0)
    df["zone_count"] = df["zone"].astype("string").fillna("Missing").map(zone_counts).fillna(0)
    df["metro_count"] = df["metro"].astype("string").fillna("Missing").map(metro_counts).fillna(0)

## PIPELINE

## a. Imputation


- Categorical features will have missing values imputed using the most_frequent strategy.
- Numerical features will have missing values imputed using the median strategy.

In [ ]:
num_cols = X_train.select_dtypes(include = "number").columns.tolist()
cat_cols = X_train.select_dtypes(include = ["object", "category"]).columns.tolist()

In [ ]:
imputer = ColumnTransformer(transformers = [
                            ("categoricalImputer", SimpleImputer(strategy = "most_frequent"), cat_cols),
                            ("numericalImputer", SimpleImputer(strategy = "median"), num_cols)
                            ], remainder = 'passthrough', verbose_feature_names_out = False
                           ).set_output(transform = 'pandas')

## b. Scaling Numerical Features

- Numerical features are scaled using the StandardScaler

In [ ]:
scaler = ColumnTransformer(transformers = [
                            ("standardScaler", StandardScaler(), num_cols)
                            ], remainder = 'passthrough', verbose_feature_names_out = False
                          ).set_output(transform = 'pandas')

## c. Encoding Categorical Features

- operator_ref is dropped
- Other columns are OneHotEncoded

In [ ]:
encoder = ColumnTransformer(transformers = [
                            ("oneHot", OneHotEncoder(drop = 'first', sparse_output = False, handle_unknown = 'ignore'), cat_cols)
                            ], remainder = 'passthrough', verbose_feature_names_out = False
                           ).set_output(transform = 'pandas')

In [ ]:
pre_pipeline = Pipeline(steps = [
    ("Imputation", imputer), ("Encoding", encoder), ("Scaling", scaler)
]).set_output(transform = 'pandas')

pre_pipeline

In [ ]:
# Interaction features between max_seats and space_class encoded columns.
def add_capacity_interactions(X):
    space_class_cols = [c for c in X.columns if c.startswith("space_class_")]

    interactions = pd.DataFrame(
        X["max_seats"].to_numpy()[:, None] * X[space_class_cols].to_numpy(),
        columns = [f"max_seats_x_{c}" for c in space_class_cols],
        index = X.index
    )

    return pd.concat([X, interactions], axis = 1)

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer
from sklearn.feature_selection import RFE
from sklearn.ensemble import RandomForestClassifier

full_pipeline = Pipeline(steps=[
    ("Preprocessing", pre_pipeline),

    ("Float32", FunctionTransformer(
        lambda X: X.astype(np.float32),
        validate=False
    )),

    ("Interactions", FunctionTransformer(add_capacity_interactions, validate = False)),

    ("RFE", RFE(estimator=RandomForestClassifier(n_estimators = 100, random_state = 42, class_weight = "balanced", n_jobs = 1),
                n_features_to_select = 500, step = 0.2
    ))
])

# 4. MODEL TRAINING

In [ ]:
X_train_processed = full_pipeline.fit_transform(X_train, y_train)
X_val_processed = full_pipeline.transform(X_val)
X_test_processed = full_pipeline.transform(X_test)

## Linear Models

In [ ]:
logistic_model = LogisticRegression(class_weight = "balanced")
logistic_model.fit(X_train_processed, y_train)
logistic_pred = logistic_model.predict(X_val_processed)
logistic_score = f1_score(y_val, logistic_pred, average = "macro")

## Other non-Linear, non-Tree Models

In [ ]:
from sklearn.neural_network import MLPClassifier

mlp_model = MLPClassifier(
        hidden_layer_sizes = (64,),
        activation = "relu",
        solver = "adam",
        batch_size = 1024,
        max_iter = 30,
        early_stopping = True,
        n_iter_no_change = 5,
        random_state = 42,
        verbose = True
    )

mlp_model.fit(X_train_processed, y_train)
mlp_pred = mlp_model.predict(X_val_processed)
mlp_score = f1_score(y_val, mlp_pred, average = "macro")

## Decision Tree

In [ ]:
from sklearn.tree import DecisionTreeClassifier

dt_model = DecisionTreeClassifier(random_state = 42, class_weight = "balanced")
dt_model.fit(X_train_processed, y_train)
dt_pred = dt_model.predict(X_val_processed)
dt_score = f1_score(y_val, dt_pred, average = "macro")

## Bagging (Random Forest)

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(n_estimators = 300, max_features = 0.15, random_state = 42, class_weight = "balanced")
rf_model.fit(X_train_processed, y_train)
rf_pred = rf_model.predict(X_val_processed)
rf_score = f1_score(y_val, rf_pred, average="macro")

## Boosting

In [ ]:
from sklearn.ensemble import AdaBoostClassifier

boost_model = AdaBoostClassifier(random_state = 42)
boost_model.fit(X_train_processed, y_train)
boost_pred = boost_model.predict(X_val_processed)
boost_score = f1_score(y_val, boost_pred, average="macro")

In [ ]:
models = ["Logistic", "MLP", "DT", "Bagging (Random Forest)", "Boosting"]
scores = [logistic_score, mlp_score, dt_score, rf_score, boost_score]

print(dict(zip(models, scores)))

sns.barplot(x = models, y = scores)
plt.xlabel("Models")
plt.xticks(rotation = 45, ha = 'right')
plt.ylabel("Macro-F1")
plt.title("Macro-F1 Score vs Model on Validation Set")

# 5. HYPERPARAMETER TUNING

In [ ]:
from sklearn.model_selection import StratifiedKFold

cv = StratifiedKFold(n_splits = 3, shuffle = True, random_state = 42)

## Logistic Regression

In [ ]:
from scipy.stats import loguniform

logistic_params = {
    "C": loguniform(1e-3, 1e2),
    "class_weight": [None, "balanced"]
}

logistic_search = RandomizedSearchCV(
    estimator = logistic_model,
    param_distributions = logistic_params,
    n_iter = 5,
    scoring = "f1_macro",
    cv = cv,
    n_jobs = 1,
    pre_dispatch = 1,
    random_state = 42,
    verbose = 1
)

logistic_search.fit(X_train_processed, y_train)
logistic_search_pred = logistic_search.predict(X_val_processed)
logistic_hpt_score = f1_score(y_val, logistic_search_pred, average = "macro")

## MLP

In [ ]:
mlp_params = {
    "hidden_layer_sizes": [(64,), (128,), (64, 32), (128, 64)],
    "activation": ["relu", "tanh"],
    "alpha": loguniform(1e-5, 1e-2),
    "learning_rate_init": loguniform(1e-4, 1e-2)
}

mlp_search = RandomizedSearchCV(
    estimator = mlp_model,
    param_distributions = mlp_params,
    n_iter = 3,
    scoring = "f1_macro",
    cv = cv,
    n_jobs = 1,
    random_state = 42,
    verbose = 1
)

mlp_search.fit(X_train_processed, y_train)
mlp_search_pred = mlp_search.predict(X_val_processed)
mlp_hpt_score = f1_score(y_val, mlp_search_pred, average="macro")

## Boosting

In [ ]:
boost_params = {
    "n_estimators": [25, 50, 100],
    "learning_rate": [0.05, 0.1, 0.5, 1.0]
}

boost_search = RandomizedSearchCV(
    estimator = boost_model,
    param_distributions = boost_params,
    n_iter = 3,
    scoring = "f1_macro",
    cv = StratifiedKFold(n_splits=2, shuffle=True, random_state=42),
    n_jobs = 1,
    pre_dispatch = 1,
    random_state = 42,
    verbose = 1
)

boost_search.fit(X_train_processed, y_train)
boost_search_pred = boost_search.predict(X_val_processed)
boost_hpt_score = f1_score(y_val, boost_search_pred, average="macro")

# 6. VOTING AND STACKING CLASSIFIERS

In [ ]:
from sklearn.ensemble import VotingClassifier, StackingClassifier
from sklearn.metrics import f1_score
from sklearn.linear_model import LogisticRegression

ensemble_models = [
    ("rf", rf_model),
    ("mlp", mlp_search.best_estimator_)
]

voting_model = VotingClassifier(estimators = ensemble_models, voting = "soft", n_jobs = 1)

voting_model.fit(X_train_processed, y_train)

voting_pred = voting_model.predict(X_val_processed)
voting_hpt_score = f1_score(y_val, voting_pred, average="macro")

In [ ]:
stacking_model = StackingClassifier(
    estimators = ensemble_models,
    final_estimator = LogisticRegression(max_iter = 1000, class_weight = "balanced"),
    cv = 2,
    n_jobs = 1
)

stacking_model.fit(X_train_processed, y_train)

stacking_pred = stacking_model.predict(X_val_processed)
stacking_hpt_score = f1_score(y_val, stacking_pred, average="macro")

In [ ]:
models = ["Random Forest (Baseline)", "MLP (Tuned)", "Logistic Regression (Tuned)", "AdaBoost (Tuned)", "Voting", "Stacking"]
scores = [rf_score, mlp_hpt_score, logistic_hpt_score, boost_hpt_score, voting_hpt_score, stacking_hpt_score]

print(dict(zip(models, scores)))

sns.barplot(x = models, y = scores)
plt.xlabel("Models")
plt.xticks(rotation = 45, ha = 'right')
plt.ylabel("Macro-F1")
plt.title("Macro-F1 Score vs Model on Validation Set")

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

print(classification_report(y_val, stacking_pred, digits = 4))
print(confusion_matrix(y_val, stacking_pred))

# 7. FINAL SUBMISSION

In [ ]:
import gc
import numpy as np

# Combine training and validation data
X_final = pd.concat([X_train, X_val], ignore_index=True)
y_final = pd.concat([y_train, y_val]).to_numpy()

# Recalculate group frequencies using ALL labelled data
class_counts = X_final["space_class"].fillna("Missing").value_counts()
zone_counts = X_final["zone"].astype("string").fillna("Missing").value_counts()
metro_counts = X_final["metro"].astype("string").fillna("Missing").value_counts()

def update_group_counts(df):
    return df.assign(
        class_count=df["space_class"].fillna("Missing").map(class_counts).fillna(0),
        zone_count=df["zone"].astype("string").fillna("Missing").map(zone_counts).fillna(0),
        metro_count=df["metro"].astype("string").fillna("Missing").map(metro_counts).fillna(0)
    )

X_final = update_group_counts(X_final)
X_test = update_group_counts(X_test)

# Garbage
del X_train, X_val
gc.collect()

X_final_processed = full_pipeline.fit_transform(X_final, y_final)
X_test_processed = full_pipeline.transform(X_test)

# Garbage
del X_test, X_final
gc.collect()

# Train Final model
stacking_model.fit(X_final_processed, y_final)
y_pred = stacking_model.predict(X_test_processed)

# Garbage
del X_final_processed, X_test_processed, y_final
gc.collect()

In [ ]:
submission = pd.DataFrame({
    "ID": range(len(y_pred)),
    "price_tier": y_pred
})

submission.to_csv("submission.csv", index = False)